In [1]:
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(dotenv_path=Path(".env"))
huggingface_token = os.environ.get("HUGGINGFACE_TOKEN")

if not huggingface_token:
    raise ValueError("HUGGINGFACE_TOKEN 이 .env 파일에 없습니다.")

login(token=huggingface_token)


/home/mobile/jaeyoung/all-in-one-llm-book/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import wandb

from sklearn.model_selection import train_test_split

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    pipeline,
    Trainer,
)
from transformers.integrations import WandbCallback
from trl import DataCollatorForCompletionOnlyLM
import evaluate

# 모델과 토크나이저 불러오기
model_name = "google/gemma-2b-it"
use_cuda = torch.cuda.is_available()
use_bf16 = use_cuda and torch.cuda.is_bf16_supported()
model_dtype = torch.bfloat16 if use_bf16 else (torch.float16 if use_cuda else torch.float32)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=model_dtype,
)

if use_cuda:
    model = model.to("cuda")

tokenizer = AutoTokenizer.from_pretrained(model_name)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 164/164 [00:00<00:00, 2246.20it/s]
